# Example 2: Agentic Research Planner with MCP-Backed Sources
Companion notebook for **Example 2** of Chapter 6.

A broad goal gets decomposed into subquestions, each subquestion is routed to the right MCP-backed source (documents, a supplier database, or web-style search), and a synthesis step merges the findings and flags coverage gaps. Self-contained -- run this notebook on its own.

In [1]:
from typing import TypedDict


class ResearchState(TypedDict, total=False):
    user_goal: str
    plan: list[str]
    question_index: int
    current_question: str
    findings: list[dict]
    contradictions: list[str]
    gaps: list[str]
    next_action: str
    final_answer: str

`question_index` isn't in the book's original state sketch -- it's added here so the graph can loop through *all* of `plan`'s subquestions, not just one, matching the chapter's description: "send those questions to different sources."

**Planner node** -- turns one broad goal into focused subquestions:

In [2]:
def plan_research(state: ResearchState) -> ResearchState:
    return {
        "plan": [
            "Which battery materials have the highest supply risk?",
            "What policy or trade risks affect battery imports?",
            "Which EV makers are most exposed to supplier concentration?",
        ],
        "question_index": 0,
        "findings": [],
        "contradictions": [],
        "gaps": [],
    }

In [3]:
def select_next_question(state: ResearchState) -> ResearchState:
    """Pop the next question off the plan, or signal that synthesis should run."""
    plan = state["plan"]
    index = state["question_index"]
    if index >= len(plan):
        return {"next_action": "synthesize"}
    return {"current_question": plan[index], "next_action": "route"}

**Conditional routing** -- a document server answers policy questions, a database server provides supplier metrics, and a search-like tool fills recent market gaps:

In [4]:
def route_source(state: ResearchState) -> ResearchState:
    q = state["current_question"].lower()
    if "policy" in q or "trade" in q:
        return {"next_action": "docs"}
    if "supplier" in q or "metrics" in q:
        return {"next_action": "db"}
    return {"next_action": "search"}

## A small MCP server for the three sources
One server, three tools -- `search_documents`, `query_supplier_database`, `search_market_web` -- each returning deterministic mock evidence, no API keys needed.

In [5]:
%%writefile research_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("ResearchServer")


@mcp.tool()
def search_documents(query: str) -> dict:
    """Search policy and industry documents for a research question."""
    q = query.lower()
    if "policy" in q or "trade" in q:
        hits = ["Battery Import Policy Outlook 2025: trade controls may raise import costs.",
                "Critical Minerals Policy Brief: export restrictions affect some regions."]
    else:
        hits = ["EV Supply Chain Notes: recent policy and sourcing changes affect planning."]
    return {"query": query, "hits": hits, "source_type": "documents"}


@mcp.tool()
def query_supplier_database(query: str) -> dict:
    """Return supplier concentration metrics for EV makers."""
    rows = [
        {"company": "EV Maker A", "top_supplier_share": 0.58, "risk": "high"},
        {"company": "EV Maker B", "top_supplier_share": 0.41, "risk": "medium"},
        {"company": "EV Maker C", "top_supplier_share": 0.29, "risk": "lower"},
    ]
    return {"query": query, "rows": rows, "source_type": "database"}


@mcp.tool()
def search_market_web(query: str) -> dict:
    """Search recent market updates relevant to a research question."""
    results = [
        "Battery raw material prices remain volatile due to shipping delays.",
        "EV makers diversify sourcing with second-source supplier contracts.",
    ]
    return {"query": query, "results": results, "source_type": "search"}


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing research_server.py


## Retrieval nodes
The book shows `query_docs`; `query_db` and `query_search` follow the exact same shape for the other two routes. Each writes **structured** evidence into `findings`, not raw text.

In [6]:
import sys, os

# Jupyter's stdout/stderr streams don't expose a real OS file descriptor, which crashes
# the MCP SDK's Windows subprocess-spawning fallback (it isn't needed outside Jupyter).
if os.name == "nt":
    try:
        sys.stderr.fileno()
    except Exception:
        sys.stderr = os.fdopen(os.dup(2), "w")

import asyncio
import json
import sys
from pathlib import Path
from typing import Any

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langgraph.graph import StateGraph, START, END

async def call_tool(session: ClientSession, name: str, arguments: dict) -> Any:
    """Call an MCP tool and unwrap its result into a plain Python value."""
    result = await session.call_tool(name, arguments)
    if result.structuredContent is not None:
        content = result.structuredContent
        return content.get("result", content) if isinstance(content, dict) else content
    text = result.content[0].text
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return text

In [7]:
async def query_docs(state: ResearchState, mcp: ClientSession) -> ResearchState:
    result = await call_tool(mcp, "search_documents", {"query": state["current_question"]})
    return {"findings": state["findings"] + [{"question": state["current_question"], "source": "docs", "evidence": result}]}


async def query_db(state: ResearchState, mcp: ClientSession) -> ResearchState:
    result = await call_tool(mcp, "query_supplier_database", {"query": state["current_question"]})
    return {"findings": state["findings"] + [{"question": state["current_question"], "source": "db", "evidence": result}]}


async def query_search(state: ResearchState, mcp: ClientSession) -> ResearchState:
    result = await call_tool(mcp, "search_market_web", {"query": state["current_question"]})
    return {"findings": state["findings"] + [{"question": state["current_question"], "source": "search", "evidence": result}]}


def advance_question(state: ResearchState) -> ResearchState:
    """Move to the next subquestion once the current one has evidence."""
    return {"question_index": state["question_index"] + 1}

**Synthesis node** -- merges results and records missing coverage (the book describes this step but leaves the code to the reader):

In [8]:
def synthesize_findings(state: ResearchState) -> ResearchState:
    findings = state["findings"]
    sources_used = {item["source"] for item in findings}
    gaps = list(state["gaps"])
    for src in ("docs", "db", "search"):
        if src not in sources_used:
            gaps.append(f"No {src} evidence collected")

    final_answer = (
        f"Reviewed {len(findings)} findings across {len(sources_used)} sources. "
        "Supplier concentration and policy/trade friction are the clearest risk drivers "
        "for EV battery supply chains."
    )
    if gaps:
        final_answer += " Remaining gaps: " + "; ".join(gaps) + "."
    return {"gaps": gaps, "final_answer": final_answer}

## Wiring the graph
A loop: `select_next_question` <-> `route_source` -> one of the three retrieval nodes -> `advance_question` -> back to `select_next_question`, until the plan is exhausted.

In [9]:
def build_graph(mcp_session: ClientSession):
    graph = StateGraph(ResearchState)

    async def query_docs_node(state):
        return await query_docs(state, mcp_session)

    async def query_db_node(state):
        return await query_db(state, mcp_session)

    async def query_search_node(state):
        return await query_search(state, mcp_session)

    graph.add_node("plan_research", plan_research)
    graph.add_node("select_next_question", select_next_question)
    graph.add_node("route_source", route_source)
    graph.add_node("query_docs", query_docs_node)
    graph.add_node("query_db", query_db_node)
    graph.add_node("query_search", query_search_node)
    graph.add_node("advance_question", advance_question)
    graph.add_node("synthesize_findings", synthesize_findings)

    graph.add_edge(START, "plan_research")
    graph.add_edge("plan_research", "select_next_question")
    graph.add_conditional_edges(
        "select_next_question", lambda s: s["next_action"],
        {"route": "route_source", "synthesize": "synthesize_findings"},
    )
    graph.add_conditional_edges(
        "route_source", lambda s: s["next_action"],
        {"docs": "query_docs", "db": "query_db", "search": "query_search"},
    )
    graph.add_edge("query_docs", "advance_question")
    graph.add_edge("query_db", "advance_question")
    graph.add_edge("query_search", "advance_question")
    graph.add_edge("advance_question", "select_next_question")
    graph.add_edge("synthesize_findings", END)
    return graph.compile()

In [10]:
server_path = Path.cwd() / "research_server.py"
params = StdioServerParameters(command=sys.executable, args=[str(server_path)])

async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        app = build_graph(session)
        result = await app.ainvoke({"user_goal": "Compare battery supply-chain risks for EV makers in 2025"})

print("Plan:")
for q in result["plan"]:
    print(f"  - {q}")

print("\nSources used:")
for f in result["findings"]:
    print(f"  - {f['source']}: {f['question']}")

print("\nFinal answer:")
print(result["final_answer"])

Plan:
  - Which battery materials have the highest supply risk?
  - What policy or trade risks affect battery imports?
  - Which EV makers are most exposed to supplier concentration?

Sources used:
  - search: Which battery materials have the highest supply risk?
  - docs: What policy or trade risks affect battery imports?
  - db: Which EV makers are most exposed to supplier concentration?

Final answer:
Reviewed 3 findings across 3 sources. Supplier concentration and policy/trade friction are the clearest risk drivers for EV battery supply chains.
